# nn.Module、参数与 logits

## 学习目标

能够定义模块、检查参数注册、追踪前向形状，并正确理解分类 logits。


## 概念模型与执行路径

`nn.Module` 管理子模块、参数、设备移动和训练模式。调用 `model(x)` 会执行 hooks 后进入 `forward`。分类头输出未归一化 logits，交叉熵内部完成 log-softmax。


### 实验 1


In [ ]:
import torch
from torch import nn

class Classifier(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8, classes=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, classes))

    def forward(self, inputs):
        return self.net(inputs)

model = Classifier()
print(model)


### 实验 2


In [ ]:
batch = torch.randn(5, 4)
logits = model(batch)
print("input -> logits:", batch.shape, "->", logits.shape)
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape), parameter.requires_grad)


### 实验 3


In [ ]:
labels = torch.tensor([0, 2, 1, 0, 2])
loss = nn.CrossEntropyLoss()(logits, labels)
probabilities = logits.softmax(dim=1)
print("loss:", loss.item())
print("probability row sums:", probabilities.sum(dim=1))


### 实验 4


In [ ]:
activations = {}
handle = model.net[0].register_forward_hook(lambda module, args, output: activations.update(linear=output.detach()))
_ = model(batch)
handle.remove()
print("captured hidden shape:", activations["linear"].shape)


## 底层机制

只有赋值为 Module 属性的 `Parameter` 才会被注册。把层放进普通 Python list 会导致参数不出现在 `model.parameters()` 中，应使用 `ModuleList` 或 `Sequential`。


## 检查点

计算该模型参数量，并说明第一层权重为什么是 `(8, 4)` 而不是 `(4, 8)`。


## 试一试

增加一个隐藏层并用 hook 记录每层输出形状；确认最终 logits 仍为 `(5, 3)`。


## 常见错误与调试

手动 softmax 后再用交叉熵、在 `forward` 中临时创建带参数层、忘记调用 `super().__init__()`、标签 dtype 不是 long。
